# M4/M5: CIFAR-100 精度改善実験

## このノートブックでできること
- ベースライン CNN に改善手法を 1 つずつ追加して効果を比較します
- **Dropout** / **BatchNorm** / **データ拡張（Data Augmentation）** / **Adam** / **学習率スケジューリング（LR Scheduling）** の効果を体感します
- 各手法を ON/OFF した実験結果を表にまとめます

## 所要時間の目安
約 90〜120 分（実験の組み合わせを試す場合）

## 対応するサイトのモジュール
**M4: 学習のしくみ** / **M5: 過学習との戦い**（授業課題2の精度改善に直結）

## 実行環境
- **GPU ランタイム必須**。Colab メニュー「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選択してください。


## 1. ライブラリのインポートと設定

基本設定をまとめて定義します。`CONFIG` 辞書の各フラグを `True/False` に切り替えて実験します。


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy

# 乱数シードの固定（再現性のため）
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")


## 2. 実験設定（CONFIG）

以下の `CONFIG` 辞書で各手法の ON/OFF を切り替えてください。実験ごとにここを変更し、結果を記録していきましょう。


In [ ]:
# ============================================================
# 実験設定（ここを変更して各手法の効果を比較）
# ============================================================
CONFIG = {
    # データ拡張 (Data Augmentation)
    "use_random_horizontal_flip": True,   # 左右反転
    "use_random_crop": True,              # ランダムクロップ
    "use_random_erasing": False,          # ランダム消去（RandomErasing）

    # 正則化 (Regularization)
    "use_dropout": True,                  # ドロップアウト（Dropout）
    "dropout_rate": 0.3,                  # ドロップアウト率
    "use_batchnorm": True,                # バッチ正規化（BatchNorm）

    # オプティマイザ (Optimizer)
    "optimizer": "SGD",                   # "SGD" または "Adam"
    "learning_rate": 0.1,                 # 初期学習率
    "momentum": 0.9,                      # SGD 用モメンタム
    "weight_decay": 5e-4,                 # 重み減衰

    # 学習率スケジューラ (LR Scheduler)
    "scheduler": "StepLR",               # "StepLR", "CosineAnnealing", "None"
    "step_size": 10,                      # StepLR: 何エポックごとに下げるか
    "gamma": 0.1,                         # StepLR: 下げる割合

    # 学習パラメータ
    "num_epochs": 30,
    "batch_size": 128,
}

print("現在の設定:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 3. データ拡張（Data Augmentation）の定義

データ拡張は「1枚の画像から変形バリエーションを作る」技法です。これにより訓練データが実質的に増え、モデルの汎化性能（Generalization）が上がります。

機械工学で例えると、「測定データに意図的にノイズを加えてロバストなモデルを学習する」ようなものです。


In [ ]:
def build_transforms(config):
    """CONFIG に応じたデータ変換パイプラインを構築する。"""
    CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
    CIFAR100_STD  = (0.2673, 0.2564, 0.2762)

    # 訓練データの変換
    train_transforms = []

    if config["use_random_crop"]:
        # ランダムクロップ: 画像をパディングして、ランダムにクロップ
        train_transforms.append(transforms.RandomCrop(32, padding=4))
        print("  ✓ RandomCrop(32, padding=4) を追加")

    if config["use_random_horizontal_flip"]:
        # 左右反転: 50% の確率で画像を水平反転
        train_transforms.append(transforms.RandomHorizontalFlip())
        print("  ✓ RandomHorizontalFlip を追加")

    train_transforms.append(transforms.ToTensor())
    train_transforms.append(transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD))

    if config["use_random_erasing"]:
        # ランダム消去: 画像の一部をランダムにマスクする
        train_transforms.append(transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)))
        print("  ✓ RandomErasing を追加")

    transform_train = transforms.Compose(train_transforms)

    # テストデータは変換なし（評価の公平性のため）
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    ])

    return transform_train, transform_test


print("データ拡張の設定:")
transform_train, transform_test = build_transforms(CONFIG)


In [ ]:
# データセットのロード
trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2
)

print(f"訓練データ: {len(trainset):,} 枚  テストデータ: {len(testset):,} 枚")


## 4. 改善版 CNN の定義

ベースライン CNN に Dropout と BatchNorm を追加した柔軟なモデルです。`CONFIG` の `use_dropout` / `use_batchnorm` フラグで ON/OFF できます。

### Dropout とは
学習中にランダムにニューロンを「切断」して、特定のニューロンへの依存を防ぐ正則化手法です。「複数の弱いネットワークのアンサンブル（多数決）」という直感があります。

### BatchNorm とは
ミニバッチ内でチャンネルごとに平均・分散を正規化する手法です。機械工学の「センサ信号の正規化処理」に相当します。
これにより学習が安定し、高い学習率が使えるようになります。


In [ ]:
class ImprovedCNN(nn.Module):
    """Dropout と BatchNorm を選択的に使える改善版 CNN。"""

    def __init__(self, num_classes=100, use_dropout=True,
                 dropout_rate=0.3, use_batchnorm=True):
        super().__init__()
        self.use_batchnorm = use_batchnorm
        self.use_dropout = use_dropout

        def conv_block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(2, 2))
            return layers

        self.features = nn.Sequential(
            *conv_block(3,   64),   # 64 × 16 × 16
            *conv_block(64, 128),   # 128 × 8 × 8
            *conv_block(128, 256),  # 256 × 4 × 4
        )

        classifier_layers = [
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            classifier_layers.append(nn.Dropout(p=dropout_rate))
        classifier_layers += [
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            classifier_layers.append(nn.Dropout(p=dropout_rate))
        classifier_layers.append(nn.Linear(512, num_classes))

        self.classifier = nn.Sequential(*classifier_layers)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# モデルを CONFIG に従って構築
model = ImprovedCNN(
    num_classes=100,
    use_dropout=CONFIG["use_dropout"],
    dropout_rate=CONFIG["dropout_rate"],
    use_batchnorm=CONFIG["use_batchnorm"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"モデルのパラメータ数: {total_params:,}")
print(f"Dropout: {'ON (rate=' + str(CONFIG['dropout_rate']) + ')' if CONFIG['use_dropout'] else 'OFF'}")
print(f"BatchNorm: {'ON' if CONFIG['use_batchnorm'] else 'OFF'}")


## 5. オプティマイザと学習率スケジューラの設定

`CONFIG` の設定に応じて SGD または Adam を選び、学習率スケジューラも切り替えます。

- **SGD + Momentum**: 古典的で安定。初期学習率を高めに設定できる
- **Adam**: 適応学習率。初期収束が速い。`lr=0.001` 程度が典型的


In [ ]:
def build_optimizer_scheduler(model, config, num_iterations):
    """CONFIG に応じたオプティマイザとスケジューラを構築する。"""

    if config["optimizer"] == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=config["learning_rate"] if config["learning_rate"] < 0.01 else 0.001,
            weight_decay=config["weight_decay"]
        )
        print(f"  オプティマイザ: Adam (lr={optimizer.param_groups[0]['lr']:.4f})")
    else:  # SGD
        optimizer = optim.SGD(
            model.parameters(),
            lr=config["learning_rate"],
            momentum=config["momentum"],
            weight_decay=config["weight_decay"]
        )
        print(f"  オプティマイザ: SGD (lr={config['learning_rate']}, momentum={config['momentum']})")

    if config["scheduler"] == "StepLR":
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=config["step_size"], gamma=config["gamma"]
        )
        print(f"  スケジューラ: StepLR (step={config['step_size']}, gamma={config['gamma']})")
    elif config["scheduler"] == "CosineAnnealing":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config["num_epochs"]
        )
        print(f"  スケジューラ: CosineAnnealingLR (T_max={config['num_epochs']})")
    else:
        scheduler = None
        print("  スケジューラ: なし")

    return optimizer, scheduler


print("オプティマイザ設定:")
optimizer, scheduler = build_optimizer_scheduler(
    model, CONFIG, len(trainloader)
)
criterion = nn.CrossEntropyLoss()


## 6. 学習の実行

学習ループを実行します。`CONFIG` で設定したエポック数分学習します。


In [ ]:
def run_training(model, trainloader, testloader, criterion, optimizer, scheduler, config):
    """学習ループを実行し、履歴を返す。"""
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "lr": []}

    print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Test Loss':>9} {'Test Acc':>8} {'LR':>8}")
    print("-" * 58)

    for epoch in range(1, config["num_epochs"] + 1):
        # --- 訓練 ---
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * images.size(0)
            tr_correct += outputs.max(1)[1].eq(labels).sum().item()
            tr_total += images.size(0)

        # --- 評価 ---
        model.eval()
        te_loss, te_correct, te_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                te_loss += loss.item() * images.size(0)
                te_correct += outputs.max(1)[1].eq(labels).sum().item()
                te_total += images.size(0)

        current_lr = optimizer.param_groups[0]['lr']
        if scheduler:
            scheduler.step()

        # 記録
        history["train_loss"].append(tr_loss / tr_total)
        history["train_acc"].append(100.0 * tr_correct / tr_total)
        history["test_loss"].append(te_loss / te_total)
        history["test_acc"].append(100.0 * te_correct / te_total)
        history["lr"].append(current_lr)

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>5} {history['train_loss'][-1]:>10.4f} "
                  f"{history['train_acc'][-1]:>8.2f}% "
                  f"{history['test_loss'][-1]:>9.4f} "
                  f"{history['test_acc'][-1]:>7.2f}% "
                  f"{current_lr:>8.5f}")

    best_acc = max(history["test_acc"])
    best_epoch = history["test_acc"].index(best_acc) + 1
    print(f"\n最高テスト精度: {best_acc:.2f}% (Epoch {best_epoch})")
    return history


history = run_training(model, trainloader, testloader, criterion, optimizer, scheduler, CONFIG)


## 7. 学習曲線の可視化と過学習の確認

訓練精度（Train Accuracy）とテスト精度（Test Accuracy）の乖離が**過学習（Overfitting）** の指標です。Dropout や BatchNorm によってどれだけ差が縮まるか確認しましょう。


In [ ]:
def plot_history(history, title="学習曲線"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="royalblue")
    axes[0].plot(epochs, history["test_loss"],  label="Test Loss",  color="tomato")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("損失曲線")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="royalblue")
    axes[1].plot(epochs, history["test_acc"],  label="Test Acc",  color="tomato")
    # 乖離を塗りつぶし（過学習の可視化）
    axes[1].fill_between(
        epochs, history["train_acc"], history["test_acc"],
        alpha=0.15, color="orange", label="Gap (Overfitting)"
    )
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("精度曲線（オレンジ = 過学習ギャップ）")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    final_gap = history["train_acc"][-1] - history["test_acc"][-1]
    print(f"最終エポックの過学習ギャップ: {final_gap:.2f}%")
    print(f"  訓練精度: {history['train_acc'][-1]:.2f}%  テスト精度: {history['test_acc'][-1]:.2f}%")


plot_history(history, title="改善版 CNN の学習結果")


## 8. 実験結果の記録

実験設定と結果を辞書に記録します。複数の実験を比較するために使います。


In [ ]:
# 実験結果の記録
experiment_log = []

def record_experiment(config, history, name=""):
    """実験結果を記録する。"""
    result = {
        "name": name,
        "config": deepcopy(config),
        "best_test_acc": max(history["test_acc"]),
        "final_test_acc": history["test_acc"][-1],
        "final_train_acc": history["train_acc"][-1],
        "overfitting_gap": history["train_acc"][-1] - history["test_acc"][-1],
    }
    experiment_log.append(result)
    return result

# 現在の実験を記録
exp_name = (
    f"DA={'ON' if CONFIG['use_random_crop'] else 'OFF'}_"
    f"DO={'ON' if CONFIG['use_dropout'] else 'OFF'}_"
    f"BN={'ON' if CONFIG['use_batchnorm'] else 'OFF'}_"
    f"Opt={CONFIG['optimizer']}_"
    f"Sch={CONFIG['scheduler']}"
)
res = record_experiment(CONFIG, history, name=exp_name)

print(f"実験名: {exp_name}")
print(f"最高テスト精度: {res['best_test_acc']:.2f}%")
print(f"過学習ギャップ: {res['overfitting_gap']:.2f}%")


## 9. 実験比較のまとめ

複数の実験が記録されたら、以下のセルで比較表を表示します。上の `CONFIG` を変えて「セル 2（CONFIG定義）から再実行」を繰り返してください。

**推奨実験順序:**
1. ベースライン: DA=OFF, DO=OFF, BN=OFF, Opt=SGD, Sch=StepLR
2. BN のみ追加: BN=ON
3. Dropout 追加: DO=ON
4. データ拡張追加: DA=ON
5. Adam に変更: Opt=Adam
6. CosineAnnealing: Sch=CosineAnnealing
7. すべて ON


In [ ]:
# 比較表の表示
if len(experiment_log) > 0:
    print(f"{'実験名':<55} {'Best Acc':>8} {'Final Acc':>9} {'Gap':>6}")
    print("-" * 82)
    for r in experiment_log:
        print(f"{r['name']:<55} {r['best_test_acc']:>7.2f}% "
              f"{r['final_test_acc']:>8.2f}% {r['overfitting_gap']:>5.2f}%")
else:
    print("まだ実験記録がありません。上のセルを実行してください。")


## 10. 試してみよう（課題）

### 課題 1: Dropout 率を変えてみる
- `dropout_rate` を `0.1`, `0.3`, `0.5` で試して、  過学習ギャップと最終精度の変化を記録してください。
- 高すぎる Dropout 率は何を引き起こしますか？

### 課題 2: スケジューラを比較する
- `"StepLR"` と `"CosineAnnealing"` を比較してください。  学習率の変化パターンと最終精度はどう違いますか？

### 課題 3: データ拡張の効果を分解する
- `use_random_crop=True` のみ、`use_random_horizontal_flip=True` のみ、  `use_random_erasing=True` のみでそれぞれ実験し、  どの拡張が最も効果的か調べてください。
